In [1]:
# ============================================
# 🚀 INSTALLATION (exécuter une seule fois dans le terminal)
# ============================================
# Ouvre un terminal et lance ces commandes :
#
# pip install openai-whisper pydub
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
#
# (cu121 = CUDA 12.1, adapte si tu as une autre version)
# ============================================

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ CUDA non détecté ! Installe PyTorch avec CUDA :")
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

✅ PyTorch: 2.6.0+cu124
✅ CUDA disponible: True
🎮 GPU: NVIDIA GeForce RTX 5060 Laptop GPU
💾 VRAM: 8.0 GB


c:\Users\thiba\Desktop\gabrielle\.venv\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


In [4]:
import whisper
import torch
import subprocess
import numpy as np
import re
import os
import math

# Utiliser ffmpeg bundlé avec imageio-ffmpeg
try:
    import imageio_ffmpeg
    FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()
    print(f"✅ FFmpeg trouvé: {FFMPEG_PATH}")
except ImportError:
    print("❌ Installe imageio-ffmpeg: pip install imageio-ffmpeg")
    raise

# ----------------- CONFIGURATION -----------------
audio_file_path = "../public/audio/gabrielle/Majorite_de_minorite.mp3"
model_name = "large-v3"  # Meilleur modèle (utilise "medium" si trop lent sur CPU)
language = "fr"
song_id = 3  # ID dans useTracks.ts
# -------------------------------------------------

def load_audio_ffmpeg(file_path, sr=16000):
    """Charge un fichier audio avec ffmpeg et retourne un numpy array."""
    cmd = [
        FFMPEG_PATH, "-i", file_path,
        "-f", "s16le",  # Format PCM 16-bit
        "-acodec", "pcm_s16le",
        "-ar", str(sr),  # Sample rate
        "-ac", "1",  # Mono
        "-loglevel", "error",
        "-"
    ]
    process = subprocess.run(cmd, capture_output=True)
    if process.returncode != 0:
        raise RuntimeError(f"FFmpeg error: {process.stderr.decode()}")
    audio = np.frombuffer(process.stdout, dtype=np.int16).astype(np.float32) / 32768.0
    return audio

# --- 1. Vérification fichier ---
if not os.path.exists(audio_file_path):
    print(f"❌ Fichier non trouvé: {audio_file_path}")
    print(f"📂 Dossier courant: {os.getcwd()}")
else:
    print(f"✅ Fichier trouvé: {audio_file_path}")

    # --- 2. Charger l'audio ---
    print("\n🔄 Chargement audio...")
    audio_np = load_audio_ffmpeg(audio_file_path)
    print(f"✅ Audio chargé: {len(audio_np)/16000:.1f} secondes")

    # --- 3. Chargement modèle ---
    device = "cpu"  # RTX 50xx pas encore supporté
    print(f"\n🚀 Chargement Whisper '{model_name}' sur {device.upper()}...")
    print("⏳ (Le modèle large-v3 fait ~3GB, patience...)")
    model = whisper.load_model(model_name, device=device)
    print(f"✅ Modèle chargé!")

    # --- 4. Transcription ---
    print("\n🎤 Transcription en cours (peut prendre quelques minutes sur CPU)...")
    result = model.transcribe(
        audio_np,
        language=language,
        word_timestamps=False,
        fp16=False,
        verbose=False,
        condition_on_previous_text=True,
        temperature=0,
    )
    print("✅ Transcription terminée!")

    # --- 5. Formatage ---
    lines = []
    for seg in result['segments']:
        time = math.floor(seg['start'])
        text = seg['text'].strip().replace("'", "\\'")
        if text and re.search(r'\w', text):
            lines.append(f"{{ time: {time}, text: '{text}' }}")

    output = f"""  {song_id}: [
    {{ time: 0, text: '...' }},
    {chr(10) + '    '.join(lines)},
  ],"""

    # --- 6. Résultat ---
    print("\n" + "="*60)
    print("   ✅ COPIER/COLLER DANS useLyrics.ts ✅")
    print("="*60)
    print(output)
    print("="*60)

✅ FFmpeg trouvé: c:\Users\thiba\Desktop\gabrielle\.venv\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
✅ Fichier trouvé: ../public/audio/gabrielle/Majorite_de_minorite.mp3

🔄 Chargement audio...
✅ Audio chargé: 308.3 secondes

🚀 Chargement Whisper 'large-v3' sur CPU...
⏳ (Le modèle large-v3 fait ~3GB, patience...)
✅ Modèle chargé!

🎤 Transcription en cours (peut prendre quelques minutes sur CPU)...


100%|██████████| 30828/30828 [04:29<00:00, 114.20frames/s]

✅ Transcription terminée!

   ✅ COPIER/COLLER DANS useLyrics.ts ✅
  3: [
    { time: 0, text: '...' },
    
{ time: 0, text: 'Aujourd\'hui, attaquons-nous à une majorité de minorités' }    { time: 21, text: 'Je pense que vous savez déjà de qui nous allons parler' }    { time: 27, text: 'Je parle bien sûr de tous ces petits fils de putes' }    { time: 34, text: 'Vivement qu\'on leur mette des culputes' }    { time: 38, text: 'Ils sont là, gueulés dans les manifs, enfin on devrait plutôt dire y\'a-le' }    { time: 44, text: 'Ça a des combats insignifiants, débiles, idiots et superficiels' }    { time: 47, text: 'Trop occupés à oublier de se raser que de réviser leur cours de SVT' }    { time: 51, text: 'S\'il vous plaît, retournez à l\'école et quittez votre minorité' }    { time: 57, text: 'Les LGBT qui a plus, c\'est grand fou' }    { time: 60, text: 'T\'es une fille quant à une zaza, t\'es un garçon quant à un zouzou' }    { time: 64, text: 'Arrêtez de nous faire chier avec tous vos g